В основе алгоритма быстрой свёртки лежит принцип соответствия свёртки вовременной области умножению в частотной области. С помощью ДПФ входнойсигнал переводится в частотную область, где он умножается на частотную характе-ристику фильтра, после чего вновь переносится во временную область с помощьюобратного ДПФ. 

In [ ]:
# БЫСТРАЯ СВЁРТКА
# Данная программа позволяет выполнить свёртку 10 000 000 отсчётов входного
# сигнала с 400 отсчётами импульсной характеристики. Входной сигнал
# разбивается на 16 000 сегментов по 625 отсчётов. Порядок БПФ равен 1024.

import numpy as np
import matplotlib.pyplot as plt

# ИНИЦИАЛИЗАЦИЯ МАССИВОВ
XX = np.zeros(1024)          # Массив отсчётов сигнала (для БПФ)
REX = np.zeros(513)          # Массив действительных частотных компонент (для БПФ)
IMX = np.zeros(513)          # Массив мнимых частотных компонент (для БПФ)
REFR = np.zeros(513)         # Массив действительных компонент частотной характеристики
IMFR = np.zeros(513)         # Массив мнимых компонент частотной характеристики
OLAP = np.zeros(399)         # Массив временного хранения перекрывающихся отсчётов

def load_impulse_response():
    """
    Подпрограмма, сохраняющая импульсную характеристику в XX
    В реальной программе здесь должен быть код загрузки ИХ
    """
    # Заглушка - заполняем тестовой импульсной характеристикой
    # Первые 400 отсчётов - ИХ, остальные - нули
    for i in range(400):
        XX[i] = np.exp(-0.01 * i) * np.sin(0.1 * i)  # Пример ИХ
    
    # Дополняем нулями до 1024 точек
    for i in range(400, 1024):
        XX[i] = 0

def compute_fft():
    """
    Подпрограмма вычисления БПФ: XX[] --> REX[] и IMX[]
    """
    # Вычисляем БПФ
    fft_result = np.fft.rfft(XX)
    
    # Разделяем на действительную и мнимую части
    global REX, IMX
    REX = np.real(fft_result)
    IMX = np.imag(fft_result)

def compute_ifft():
    """
    Подпрограмма вычисления ОБПФ: REX[] и IMX[] --> XX[]
    """
    # Объединяем действительную и мнимую части
    complex_spectrum = REX + 1j * IMX  #//?
    
    # Вычисляем обратное БПФ
    global XX
    XX = np.fft.irfft(complex_spectrum)

def load_segment(segment_num):
    """
    Подпрограмма, загружающая очередной сегмент в XX[]
    """
    # В реальной программе здесь должен быть код загрузки сегмента данных
    # segment_data = load_from_file(segment_num)  # Заглушка
    segment_data = np.random.randn(625)  # Тестовые данные
    
    # Загружаем сегмент в начало массива XX
    for i in range(625):
        XX[i] = segment_data[i]
    
    # Остальные элементы обнуляем
    for i in range(625, 1024):
        XX[i] = 0

def output_segment():
    """
    Подпрограмма вывода 625 отсчётов выходного сигнала
    """
    # В реальной программе здесь должен быть код сохранения или обработки результатов
    # save_to_file(XX[0:625])  # Заглушка
    pass

def output_overlap():
    """
    Подпрограмма вывода 399 отсчётов массива OLAP[]
    """
    # В реальной программе здесь должен быть код сохранения оставшихся отсчётов
    # save_remaining(OLAP)  # Заглушка
    pass

# ВЫЧИСЛЕНИЕ И СОХРАНЕНИЕ ЧАСТОТНОЙ ХАРАКТЕРИСТИКИ
load_impulse_response()      # Сохраняем импульсную характеристику в XX
compute_fft()                # Вычисляем БПФ

# Сохранение частотной характеристики в REFR и IMFR
for f in range(513):
    REFR[f] = REX[f]
    IMFR[f] = IMX[f]

# ПООЧЕРЁДНАЯ ОБРАБОТКА 16 000 СЕГМЕНТОВ
for segment in range(16000):
    # Загружаем очередной сегмент
    load_segment(segment)
    
    # Вычисляем БПФ сегмента
    compute_fft()
    
    # Умножение спектра сигнала на частотную характеристику
    for f in range(513):
        temp = REX[f] * REFR[f] - IMX[f] * IMFR[f]
        IMX[f] = REX[f] * IMFR[f] + IMX[f] * REFR[f]
        REX[f] = temp
    
    # Вычисляем обратное БПФ
    compute_ifft()
    
    # Сложение с отсчётами предыдущего сегмента
    for i in range(625):
        XX[i] = XX[i] + OLAP[i]
    
    # Сохранение отсчётов, образующих перекрытие
    for i in range(625, 1024):
        OLAP[i - 625] = XX[i]
    
    # Вывод 625 отсчётов выходного сигнала
    output_segment()

# Вывод 399 отсчётов массива OLAP
output_overlap()